<a href="https://colab.research.google.com/github/jawaadjariwala/Aramark/blob/main/02_bigquery_load_and_aggregations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02 — BigQuery Load & Core Aggregations

**Phase:** 1 + 2 (from the project plan)

**Input:** `gs://cs-562-aramark-project/
Andrew_Meszaros_SRF_2026-04-01-0936.csv`

**Output:** A typed BigQuery table `big-data-algorithms-493312.aramark_spend.raw_spend`

## Why we are moving to BigQuery

In Notebook 01 we demonstrated that pandas chunking works on the 44M-row dataset, but each full aggregation took ~10 minutes and had to be re-run after every Colab disconnect. That does not scale to the 10+ aggregations this project needs.

BigQuery is Google's serverless, columnar, distributed SQL warehouse. Once we load the CSV into a BigQuery table:

- Every aggregation becomes a SQL query that returns in 2–5 seconds
- The data is persistent — no re-processing on disconnects
- External datasets (weather, economic indicators) can be joined with a single SQL statement
- Power BI can connect directly, no CSV exports needed

## Step 1 — Authenticate with Google Cloud

Colab needs an auth token to call BigQuery on our behalf. We reuse the same Google account that owns the GCS bucket and the BigQuery dataset.

In [13]:
# Standard Colab authentication flow.
# A popup will ask you to pick your Google account and grant access.
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

Authenticated


In [14]:
# Check which Google account Colab is authenticated as.
!gcloud auth list

        Credentialed Accounts
ACTIVE  ACCOUNT
*       jj980@scarletmail.rutgers.edu

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



## Step 2 — Configuration

We centralize our project, dataset, and table names here so the rest of the notebook stays clean and reusable.

In [17]:
# Google Cloud project that owns both the GCS bucket and the BigQuery dataset.
PROJECT_ID = "big-data-algorithms-493312"

# BigQuery dataset we created in the console.
DATASET_ID = "aramark_spend"

# Table we will create from the CSV
TABLE_ID = "raw_spend"

GCS_URI = "gs://cs-562-aramark-project/Andrew_Meszaros_SRF_2026-04-01-0936.csv"

# BigQuery table reference (project.dataset.table)
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

print(f"Target table: {TABLE_REF}")

Target table: big-data-algorithms-493312.aramark_spend.raw_spend


## Step 3 — Load the CSV into BigQuery

We define an explicit schema rather than relying on auto-detection. Two reasons:

1. **Clean column names.** The source CSV has spaces in column names (`Year Name`, `Business Entity Type`). In SQL, those require backticks around every reference, which is error-prone and ugly. We rename everything to `snake_case` at load time.
2. **Correct types.** We want `spend_random_factor` as `FLOAT64`, `year_month` as `INTEGER`, and IDs as `STRING` to preserve leading zeros and handle nulls cleanly.

BigQuery loads the file **directly from GCS**; nothing transits through Colab. This is dramatically faster than reading chunks into pandas.

In [18]:
from google.cloud import bigquery

# Create a BigQuery client tied to our project.
client = bigquery.Client(project=PROJECT_ID)

# Explicit schema: rename all 23 columns to snake_case and assign correct types.
# The order MUST match the column order in the CSV.
schema = [
    bigquery.SchemaField("year_name",                  "INTEGER"),
    bigquery.SchemaField("year_month",                 "INTEGER"),
    bigquery.SchemaField("business_entity_type",       "STRING"),
    bigquery.SchemaField("country",                    "STRING"),
    bigquery.SchemaField("customer_market_segment_id", "STRING"),
    bigquery.SchemaField("client_id",                  "STRING"),
    bigquery.SchemaField("customer_id",                "STRING"),
    bigquery.SchemaField("customer_brand_id",          "STRING"),
    bigquery.SchemaField("customer_brand_parent_id",   "STRING"),
    bigquery.SchemaField("city",                       "STRING"),
    bigquery.SchemaField("state",                      "STRING"),
    bigquery.SchemaField("zip",                        "STRING"),   # string to preserve leading zeros
    bigquery.SchemaField("number_of_rooms",            "INTEGER"),
    bigquery.SchemaField("ecommerce_status",           "STRING"),
    bigquery.SchemaField("distributor_id",             "STRING"),   # string handles nulls cleanly
    bigquery.SchemaField("distributor_group",          "STRING"),
    bigquery.SchemaField("manufacturer_id",            "STRING"),
    bigquery.SchemaField("category_id",                "STRING"),
    bigquery.SchemaField("category_level_1",           "STRING"),
    bigquery.SchemaField("category_level_2",           "STRING"),
    bigquery.SchemaField("category_level_3",           "STRING"),
    bigquery.SchemaField("category_level_4",           "STRING"),
    bigquery.SchemaField("spend_random_factor",        "FLOAT64"),  # the spend proxy
]

# Configure the load job.
job_config = bigquery.LoadJobConfig(
    schema=schema,
    skip_leading_rows=1,                         # ignore the CSV header row
    source_format=bigquery.SourceFormat.CSV,
    write_disposition="WRITE_TRUNCATE",          # replace the table if it already exists
    allow_quoted_newlines=True,                  # safe default for CSV parsing
    max_bad_records=100,                         # tolerate up to 100 malformed rows
)

# BigQuery reads the CSV directly from GCS in parallel.
print("Starting load job — this runs server-side, takes ~2-5 minutes...")
load_job = client.load_table_from_uri(GCS_URI, TABLE_REF, job_config=job_config)

# Wait for the job to finish.
load_job.result()

print(f"Load complete. Job ID: {load_job.job_id}")

# Verify the table: row count and schema.
table = client.get_table(TABLE_REF)
print(f"Rows loaded: {table.num_rows:,}")
print(f"Columns:     {len(table.schema)}")
print(f"Size:        {table.num_bytes / 1e9:.2f} GB")


Starting load job — this runs server-side, takes ~2-5 minutes...
Load complete. Job ID: 74815932-b49f-42cb-b467-c5b845400a23
Rows loaded: 42,967,459
Columns:     23
Size:        9.43 GB


## Step 4 — Sanity Check

We re-run the Distributor Group aggregation from Notebook 01, this time in pure SQL against BigQuery. The numbers should match our earlier chunked result (MASTER DISTRIBUTOR ~$3.28B across 28.4M transactions). This confirms:

1. The load was complete (no dropped rows)
2. The types are correct (spend sums aggregate cleanly)
3. BigQuery is returning the same truth as our earlier work

And it should return in a couple of seconds rather than 10 minutes.

In [19]:
# Re-run the Distributor Group aggregation in SQL.
# Expected: top row is MASTER DISTRIBUTOR at ~$3.28B and ~28.4M transactions.
sql = f"""
SELECT
  distributor_group,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count
FROM `{TABLE_REF}`
GROUP BY distributor_group
ORDER BY total_spend DESC
LIMIT 10
"""

# .to_dataframe() runs the query and returns a pandas DataFrame so we can display it nicely.
client.query(sql).to_dataframe()

,distributor_group,total_spend,transaction_count
0,MASTER DISTRIBUTOR,3.277687e+09,28398552
1,ENGINEERING MRO,3.748054e+08,3321250
2,DIRECT BEVERAGE,3.096215e+08,1229770
3,REGIONAL PRODUCE,3.094375e+08,3035950
4,RETAIL,2.028612e+08,1698725
5,ROOM OPERATIONS - OTHER,1.950658e+08,938291
6,SPECIALTY FOODS,1.935986e+08,1064243
7,REGIONAL MEAT/POULTRY,1.891870e+08,491074
8,ENGINEERING OTHER,1.155428e+08,199098
9,None,1.065176e+08,62320


## Step 5 — What's Next

The table is loaded and validated. From this point on, every aggregation in this project is a SQL query that runs in seconds. The next sections of this notebook will compute:

- Category Level 1 and Level 2 spend distribution
- Geographic (State, City) aggregations
- Monthly time trends
- Business Entity Type (GPO vs Managed Services) comparison
- Customer Market Segment breakdown
- Number of Rooms scale bands
- Cross-dimensional views (Category × State, Distributor × State, etc.)

All of these build directly on the `raw_spend` table we just created.

## 2.1 — Category Level 1

The highest level of the product hierarchy. This answers: **what is Aramark buying the most of?**

In [20]:
# Category Level 1 spend and transaction counts.
# Shows the dominant product groups across all 43M transactions.
sql = f"""
SELECT
  category_level_1,
  ROUND(SUM(spend_random_factor), 0)                 AS total_spend,
  COUNT(*)                                           AS transaction_count,
  ROUND(AVG(spend_random_factor), 2)                 AS avg_transaction_size
FROM `{TABLE_REF}`
GROUP BY category_level_1
ORDER BY total_spend DESC
"""

cat1_df = client.query(sql).to_dataframe()
cat1_df

,category_level_1,total_spend,transaction_count,avg_transaction_size
0,FOOD,3.695921e+09,29400266,125.71
1,BEVERAGE,6.377751e+08,3568234,178.74
2,DISPOSABLES,4.154615e+08,3249175,127.87
3,MAINTENANCE AND ENGINEERING,3.816488e+08,1599542,238.60
4,CHEMICALS AND CLEANING,3.461939e+08,2009923,172.24
5,ROOM AND SPA,2.332630e+08,1321793,176.47
6,FOOD SERVICE EQUIPMENT AND SUPPLIES,1.570712e+08,912109,172.21
7,CLOTHING AND FOOTWEAR,8.192232e+07,278806,293.83
8,FURNITURE FIXTURES AND EQUIPMENT,6.722904e+07,186269,360.92
9,GOLF EQUIPMENT AND SUPPLIES,6.597244e+07,88748,743.37


## 2.2 — Category Level 2 (nested within Level 1)

Drills one level deeper. Useful for seeing what's really driving spend inside the top Level 1 categories.

In [21]:
# Category Level 1 x Level 2 — nested hierarchy view.
# Reveals which sub-categories dominate each top-level category.
sql = f"""
SELECT
  category_level_1,
  category_level_2,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count
FROM `{TABLE_REF}`
GROUP BY category_level_1, category_level_2
ORDER BY total_spend DESC
LIMIT 25
"""

cat2_df = client.query(sql).to_dataframe()
cat2_df

,category_level_1,category_level_2,total_spend,transaction_count
0,FOOD,GROCERY,672986257.0,4576689
1,FOOD,PRODUCE,637828690.0,7411234
2,FOOD,MEAT,565574425.0,2358872
3,FOOD,DAIRY,461473513.0,3986700
4,FOOD,BAKERY,441891735.0,4114643
5,DISPOSABLES,DISPOSABLES,415452447.0,3249166
6,FOOD,POULTRY,341108467.0,1286888
7,CHEMICALS AND CLEANING,CLEANING SUPPLIES,216400707.0,1784409
8,MAINTENANCE AND ENGINEERING,M&E SERVICES,161428236.0,116068
9,FOOD,SEAFOOD,158950591.0,664071


## 2.3 — Geography: State

Spend distribution across US states. Tells us where Aramark's procurement volume is concentrated geographically.

In [22]:
# State-level aggregation.
# Ranks all 50 states by total spend proxy.
sql = f"""
SELECT
  state,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count,
  COUNT(DISTINCT city)               AS unique_cities
FROM `{TABLE_REF}`
GROUP BY state
ORDER BY total_spend DESC
"""

state_df = client.query(sql).to_dataframe()
state_df

,state,total_spend,transaction_count,unique_cities
0,CA,869499953.0,5382285,681
1,TX,615573243.0,3838319,535
2,FL,583675600.0,3652381,417
3,PA,286890457.0,1619980,531
4,IL,264431571.0,2397373,515
5,NY,230677420.0,1460393,458
6,NC,210926483.0,1317593,280
7,VA,209252124.0,1543955,228
8,GA,187000556.0,1262424,224
9,OH,166533976.0,1514114,440


## 2.4 — Time Trend: Year Month

Monthly spend across the 12 months of 2025. This is our temporal signal — we'll later overlay weather and anomaly data on top of this trend.


In [23]:
# Monthly spend trend across 2025.
sql = f"""
SELECT
  year_month,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count
FROM `{TABLE_REF}`
GROUP BY year_month
ORDER BY year_month
"""

time_df = client.query(sql).to_dataframe()
time_df

,year_month,total_spend,transaction_count
0,202501,496240043.0,3520664
1,202502,484705312.0,3412601
2,202503,523134587.0,3605995
3,202504,561791457.0,3694859
4,202505,507966442.0,3627424
5,202506,431360180.0,3449008
6,202507,459145371.0,3469582
7,202508,539558919.0,3663110
8,202509,570602599.0,3763957
9,202510,610882434.0,3850342


## 2.5 — Business Entity Type

Compares the two main business models Aramark serves: **GPO** (Group Purchasing Organization customers) vs **MANAGED SERVICES** (accounts Aramark operates directly).

In [24]:
# Business Entity Type breakdown.
# GPO vs Managed Services — very different buying patterns.
sql = f"""
SELECT
  business_entity_type,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count,
  ROUND(AVG(spend_random_factor), 2) AS avg_transaction_size
FROM `{TABLE_REF}`
GROUP BY business_entity_type
ORDER BY total_spend DESC
"""

entity_df = client.query(sql).to_dataframe()
entity_df

,business_entity_type,total_spend,transaction_count,avg_transaction_size
0,GPO,3.985547e+09,34298937,116.2
1,MANAGED SERVICES,2.157629e+09,8668522,248.9


## 2.6 — Customer Market Segment

Finer-grained segmentation — healthcare, lodging, etc. Helps us understand which industries drive which spend patterns.

In [26]:
# Customer Market Segment aggregation.
sql = f"""
SELECT
  customer_market_segment_id,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count,
  COUNT(DISTINCT customer_id)        AS unique_customers
FROM `{TABLE_REF}`
GROUP BY customer_market_segment_id
ORDER BY total_spend DESC
"""

segment_df = client.query(sql).to_dataframe()
segment_df

,customer_market_segment_id,total_spend,transaction_count,unique_customers
0,MS-100003,987532994.0,3058185,2546
1,MS-100002,965033444.0,13450178,17951
2,MS-100021,895176871.0,6463213,2341
3,MS-100000,526350289.0,2632950,1268
4,MS-100022,350050512.0,2082906,612
5,MS-100001,327836530.0,967522,757
6,MS-100005,282736421.0,484589,346
7,MS-100024,282017659.0,3524454,5469
8,MS-100023,279046249.0,3745137,3616
9,MS-100004,232991200.0,1872659,1210


## 2.7 — Operational Scale: Number of Rooms

Rooms is primarily a hospitality signal. A value of 0 means non-hospitality. We bucket properties into size bands to see how procurement varies with scale.


In [27]:
# Number of Rooms banded into size categories.
# 0 rooms = non-hospitality; bands capture small/medium/large hotels.
sql = f"""
SELECT
  CASE
    WHEN number_of_rooms = 0           THEN '0 (non-hospitality)'
    WHEN number_of_rooms BETWEEN 1   AND 100  THEN '1-100'
    WHEN number_of_rooms BETWEEN 101 AND 300  THEN '101-300'
    WHEN number_of_rooms BETWEEN 301 AND 500  THEN '301-500'
    ELSE '500+'
  END AS room_band,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count,
  COUNT(DISTINCT customer_id)        AS unique_customers
FROM `{TABLE_REF}`
GROUP BY room_band
ORDER BY total_spend DESC
"""

rooms_df = client.query(sql).to_dataframe()
rooms_df


,room_band,total_spend,transaction_count,unique_customers
0,0 (non-hospitality),3.737531e+09,19215802,21747
1,101-300,1.136320e+09,12239518,10047
2,1-100,5.300996e+08,7951774,11138
3,301-500,4.444843e+08,2507448,770
4,500+,2.947416e+08,1052917,253


# Phase 2.2 — Cross-Dimensional Aggregations

Single-dimension aggregations give the headline story. Cross-cuts reveal the structure beneath that story — for example, the October peak we saw in Year Month tells us *when* Aramark spends more, but a Category × Month cut tells us *what* drives that peak.

The six cross-cuts below:

1. Category Level 1 × State — what each state buys most
2. Category Level 1 × Year Month — what drives seasonality
3. Distributor Group × State — supplier footprint by geography
4. Category Level 1 × Business Entity Type — GPO vs Managed Services buying mix
5. Distributor Group × Room Band — do large properties use different suppliers
6. Category drill-down L1 → L2 → L3 → L4 — full hierarchy for Power BI drill-through

## 2.2.1 — Category Level 1 × State

Which states lean into which categories. States differ — Hawaii is hospitality-heavy, Texas has energy infrastructure, California mixes everything. This cut exposes those differences.


In [28]:
# Top category per state, ranked by spend.
# Two-level GROUP BY; ordered by state then spend so the dominant category bubbles up.
sql = f"""
SELECT
  state,
  category_level_1,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count
FROM `{TABLE_REF}`
GROUP BY state, category_level_1
ORDER BY state, total_spend DESC
"""

cat_state_df = client.query(sql).to_dataframe()
# Preview: just the top category per state (the row at rank 1 within each state)
cat_state_df.groupby("state").head(1).reset_index(drop=True)

,state,category_level_1,total_spend,transaction_count
0,AB,MAINTENANCE AND ENGINEERING,109.0,1
1,AK,FOOD,7104351.0,46936
2,AL,FOOD,57635819.0,387545
3,AR,FOOD,25796948.0,201674
4,AZ,FOOD,102349506.0,635743
5,CA,FOOD,554277355.0,3596514
6,CO,FOOD,77488182.0,747827
7,CT,FOOD,27330184.0,199500
8,DC,FOOD,34139760.0,196090
9,DE,FOOD,14979268.0,93744


## 2.2.2 — Category Level 1 × Year Month

Which categories drive the October peak and June trough? This is critical context for the weather correlation study in Phase 5.

In [29]:
# Monthly spend, broken out by category.
# Wide format (months as rows, categories as columns) is easier to read.
sql = f"""
SELECT
  year_month,
  category_level_1,
  ROUND(SUM(spend_random_factor), 0) AS total_spend
FROM `{TABLE_REF}`
GROUP BY year_month, category_level_1
ORDER BY year_month, total_spend DESC
"""

cat_month_df = client.query(sql).to_dataframe()

# Pivot to wide format for easier visual reading: rows = months, columns = categories.
cat_month_wide = cat_month_df.pivot(
    index="year_month",
    columns="category_level_1",
    values="total_spend"
).fillna(0)

cat_month_wide

category_level_1,BEVERAGE,CHEMICALS AND CLEANING,CLOTHING AND FOOTWEAR,DISPOSABLES,FOOD,FOOD SERVICE EQUIPMENT AND SUPPLIES,FURNITURE FIXTURES AND EQUIPMENT,GOLF EQUIPMENT AND SUPPLIES,MAINTENANCE AND ENGINEERING,MEDICAL,POOL EQUIPMENT AND SUPPLIES,RETAIL AND PROMOTIONAL,ROOM AND SPA
year_month,,,,,,,,,,,,,
202501,46799845.0,28899061.0,6365749.0,33855933.0,306771588.0,11437461.0,5367404.0,2944092.0,31376965.0,2343033.0,391331.0,1487698.0,18199882.0
202502,47635437.0,25089376.0,6529087.0,32653887.0,303335430.0,11530868.0,4336262.0,3289314.0,29099515.0,2091175.0,300024.0,1828547.0,16986391.0
202503,53789702.0,26763665.0,6447971.0,35090525.0,320024799.0,13277135.0,5776038.0,5117538.0,32220011.0,2323172.0,438215.0,2487999.0,19377816.0
202504,58884051.0,32248314.0,6740740.0,37276208.0,343969754.0,13607571.0,5591691.0,4682457.0,32332706.0,2470911.0,433068.0,2746985.0,20807002.0
202505,55012330.0,28347669.0,6651252.0,34828671.0,302674478.0,13592265.0,5400273.0,4449289.0,31599948.0,2309749.0,506248.0,2404394.0,20189876.0
202506,48429472.0,27556860.0,6304459.0,30320522.0,237190987.0,12785905.0,6118593.0,4953310.0,32235822.0,2343883.0,521715.0,2182780.0,20415872.0
202507,53168964.0,29336031.0,7731831.0,33752948.0,249917275.0,13415949.0,5630706.0,4577760.0,34367735.0,2569463.0,519048.0,2381031.0,21776630.0
202508,58477386.0,31503719.0,7373416.0,36328007.0,322579284.0,14152467.0,5879528.0,4992163.0,32826649.0,2949665.0,408687.0,1914751.0,20173196.0
202509,59489514.0,28925511.0,7812457.0,37606469.0,352429147.0,14399390.0,6139862.0,6370335.0,32755281.0,3336964.0,405723.0,1832006.0,19099939.0


## 2.2.3 — Distributor Group × State

Which distributor dominates in which state? Foundation for Phase 3 concentration analysis — if one distributor supplies >80% of a state, that's a dependency risk.

In [30]:
# For each state, top distributor by spend.
# ROW_NUMBER() gives us a per-state ranking we can filter on.
sql = f"""
WITH ranked AS (
  SELECT
    state,
    distributor_group,
    ROUND(SUM(spend_random_factor), 0) AS total_spend,
    COUNT(*)                           AS transaction_count,
    ROW_NUMBER() OVER (PARTITION BY state ORDER BY SUM(spend_random_factor) DESC) AS rn
  FROM `{TABLE_REF}`
  WHERE distributor_group IS NOT NULL
  GROUP BY state, distributor_group
)
SELECT state, distributor_group, total_spend, transaction_count
FROM ranked
WHERE rn = 1
ORDER BY total_spend DESC
"""

dist_state_df = client.query(sql).to_dataframe()
dist_state_df


,state,distributor_group,total_spend,transaction_count
0,CA,MASTER DISTRIBUTOR,462537040.0,3092857
1,FL,MASTER DISTRIBUTOR,311951598.0,2331813
2,TX,MASTER DISTRIBUTOR,294462965.0,2464620
3,PA,MASTER DISTRIBUTOR,153567636.0,1048814
4,IL,MASTER DISTRIBUTOR,146609897.0,1766843
5,NC,MASTER DISTRIBUTOR,123258203.0,889662
6,VA,MASTER DISTRIBUTOR,114296443.0,1025100
7,GA,MASTER DISTRIBUTOR,108136728.0,806174
8,AZ,MASTER DISTRIBUTOR,97333852.0,610555
9,NY,MASTER DISTRIBUTOR,95721455.0,808213


## 2.2.4 — Category Level 1 × Business Entity Type

GPO and Managed Services have very different transaction sizes ($116 vs $249). This cut shows whether they also buy *different things*.

In [31]:
# Category mix within each business entity type.
# Percentages computed with a window function so we can see share within entity type.
sql = f"""
SELECT
  business_entity_type,
  category_level_1,
  ROUND(SUM(spend_random_factor), 0)                                      AS total_spend,
  ROUND(100 * SUM(spend_random_factor)
        / SUM(SUM(spend_random_factor)) OVER (PARTITION BY business_entity_type), 2)
                                                                          AS pct_of_entity_spend
FROM `{TABLE_REF}`
GROUP BY business_entity_type, category_level_1
ORDER BY business_entity_type, total_spend DESC
"""

cat_entity_df = client.query(sql).to_dataframe()
cat_entity_df

,business_entity_type,category_level_1,total_spend,pct_of_entity_spend
0,GPO,FOOD,2.392718e+09,60.03
1,GPO,BEVERAGE,2.893596e+08,7.26
2,GPO,CHEMICALS AND CLEANING,2.804065e+08,7.04
3,GPO,DISPOSABLES,2.768083e+08,6.95
4,GPO,ROOM AND SPA,2.269357e+08,5.69
5,GPO,MAINTENANCE AND ENGINEERING,2.196370e+08,5.51
6,GPO,FOOD SERVICE EQUIPMENT AND SUPPLIES,1.059281e+08,2.66
7,GPO,GOLF EQUIPMENT AND SUPPLIES,6.421791e+07,1.61
8,GPO,FURNITURE FIXTURES AND EQUIPMENT,6.118667e+07,1.54
9,GPO,CLOTHING AND FOOTWEAR,3.127107e+07,0.78


## 2.2.5 — Distributor Group × Room Band

Do large properties (500+ rooms) use different distributors than small ones? This reveals whether scale changes supplier relationships.

In [32]:
# Distributor mix within each room band.
sql = f"""
SELECT
  CASE
    WHEN number_of_rooms = 0           THEN '0 (non-hospitality)'
    WHEN number_of_rooms BETWEEN 1   AND 100  THEN '1-100'
    WHEN number_of_rooms BETWEEN 101 AND 300  THEN '101-300'
    WHEN number_of_rooms BETWEEN 301 AND 500  THEN '301-500'
    ELSE '500+'
  END AS room_band,
  distributor_group,
  ROUND(SUM(spend_random_factor), 0) AS total_spend,
  COUNT(*)                           AS transaction_count
FROM `{TABLE_REF}`
WHERE distributor_group IS NOT NULL
GROUP BY room_band, distributor_group
ORDER BY room_band, total_spend DESC
"""

dist_room_df = client.query(sql).to_dataframe()
dist_room_df

,room_band,distributor_group,total_spend,transaction_count
0,0 (non-hospitality),MASTER DISTRIBUTOR,2.274831e+09,14645978
1,0 (non-hospitality),DIRECT BEVERAGE,2.323989e+08,511952
2,0 (non-hospitality),RETAIL,1.804041e+08,932569
3,0 (non-hospitality),REGIONAL PRODUCE,1.352691e+08,1027200
4,0 (non-hospitality),SPECIALTY FOODS,1.160224e+08,318956
...,...,...,...,...
155,500+,TURF MANAGEMENT,3.356950e+05,340
156,500+,PLANT MAINTENANCE,3.028280e+05,514
157,500+,DISPOSABLES,2.318950e+05,41
158,500+,PRO SHOP,1.820560e+05,272


## 2.2.6 — Full Category Drill-Down (L1 → L2 → L3 → L4)

The complete product hierarchy. This feeds the Power BI drill-through where users click into a category to see deeper levels. We limit to the top 50 combinations to keep the preview readable — in Power BI the full table will be loaded.

In [34]:
# Full category hierarchy. L3 and L4 have some nulls; we keep them but label explicitly.
sql = f"""
SELECT
  category_level_1,
  category_level_2,
  COALESCE(category_level_3, '(none)') AS category_level_3,
  COALESCE(category_level_4, '(none)') AS category_level_4,
  ROUND(SUM(spend_random_factor), 0)   AS total_spend,
  COUNT(*)                             AS transaction_count
FROM `{TABLE_REF}`
GROUP BY category_level_1, category_level_2, category_level_3, category_level_4
ORDER BY total_spend DESC
LIMIT 50
"""

cat_drill_df = client.query(sql).to_dataframe()
cat_drill_df

,category_level_1,category_level_2,category_level_3,category_level_4,total_spend,transaction_count
0,FOOD,GROCERY,"GROCERY, OTHER","GROCERY, OTHER",109316428.0,126587
1,FOOD,MEAT,BEEF,GROUND BEEF,108623741.0,306729
2,FOOD,POULTRY,CHICKEN,CHICKEN BREASTS,99432279.0,284979
3,FOOD,GROCERY,SNACK FOODS,CHIPS,81895241.0,336390
4,FOOD,PRODUCE,POTATOES,FRENCH FRIES,66834662.0,271223
5,FOOD,MEAT,PORK,"PORK, BACON",65106900.0,224130
6,FOOD,GROCERY,PREPARED FOODS,PIZZA,62241843.0,117672
7,CHEMICALS AND CLEANING,CLEANING AND JANITORIAL SERVICES,(none),(none),61200017.0,4403
8,BEVERAGE,COFFEE,"COFFEE BEANS, GROUND","COFFEE BEANS, GROUND",60727502.0,193405
9,DISPOSABLES,DISPOSABLES,DISPOSABLE TABLEWARE,DISPOSABLE CUPS,60189559.0,446044


## Step 6 — Materialize Aggregations as BigQuery Views

Up to this point, every aggregation in this notebook ran as an inline SQL string. That works for exploration, but for Notebook 03, the Power BI dashboard, and any future work would each re-type the same SQL and eventually drift.

We fix that by wrapping each aggregation as a BigQuery **view**. A view is a stored query that behaves like a table: it always reflects the current state of `raw_spend`, costs no storage, and can be queried from any client (another notebook, the BigQuery console, Power BI's direct connector).

Naming convention: `agg_<dimension>` for single-dim rollups, `agg_<dim1>_<dim2>` for cross-cuts.


In [35]:
# Create all aggregation views. Re-running this is safe (CREATE OR REPLACE).
views = {
    # --- Single-dimension ---
    "agg_distributor_group": f"""
        SELECT
          distributor_group,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          ROUND(AVG(spend_random_factor), 2) AS avg_transaction_size
        FROM `{TABLE_REF}`
        GROUP BY distributor_group
    """,
    "agg_category_level_1": f"""
        SELECT
          category_level_1,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          ROUND(AVG(spend_random_factor), 2) AS avg_transaction_size
        FROM `{TABLE_REF}`
        GROUP BY category_level_1
    """,
    "agg_category_l1_l2": f"""
        SELECT
          category_level_1,
          category_level_2,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        GROUP BY category_level_1, category_level_2
    """,
    "agg_state": f"""
        SELECT
          state,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          COUNT(DISTINCT city)               AS unique_cities
        FROM `{TABLE_REF}`
        GROUP BY state
    """,
    "agg_year_month": f"""
        SELECT
          year_month,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        GROUP BY year_month
    """,
    "agg_business_entity_type": f"""
        SELECT
          business_entity_type,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          ROUND(AVG(spend_random_factor), 2) AS avg_transaction_size
        FROM `{TABLE_REF}`
        GROUP BY business_entity_type
    """,
    "agg_market_segment": f"""
        SELECT
          customer_market_segment_id,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          COUNT(DISTINCT customer_id)        AS unique_customers
        FROM `{TABLE_REF}`
        GROUP BY customer_market_segment_id
    """,
    "agg_room_band": f"""
        SELECT
          CASE
            WHEN number_of_rooms = 0                 THEN '0 (non-hospitality)'
            WHEN number_of_rooms BETWEEN 1   AND 100 THEN '1-100'
            WHEN number_of_rooms BETWEEN 101 AND 300 THEN '101-300'
            WHEN number_of_rooms BETWEEN 301 AND 500 THEN '301-500'
            ELSE '500+'
          END                                AS room_band,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count,
          COUNT(DISTINCT customer_id)        AS unique_customers
        FROM `{TABLE_REF}`
        GROUP BY room_band
    """,

    # --- Cross-dimensional ---
    "agg_category_state": f"""
        SELECT
          state,
          category_level_1,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        GROUP BY state, category_level_1
    """,
    "agg_category_month": f"""
        SELECT
          year_month,
          category_level_1,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        GROUP BY year_month, category_level_1
    """,
    "agg_distributor_state": f"""
        SELECT
          state,
          distributor_group,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        WHERE distributor_group IS NOT NULL
        GROUP BY state, distributor_group
    """,
    "agg_category_entity": f"""
        SELECT
          business_entity_type,
          category_level_1,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          ROUND(100 * SUM(spend_random_factor)
                / SUM(SUM(spend_random_factor)) OVER (PARTITION BY business_entity_type), 2)
                                             AS pct_of_entity_spend
        FROM `{TABLE_REF}`
        GROUP BY business_entity_type, category_level_1
    """,
    "agg_distributor_room_band": f"""
        SELECT
          CASE
            WHEN number_of_rooms = 0                 THEN '0 (non-hospitality)'
            WHEN number_of_rooms BETWEEN 1   AND 100 THEN '1-100'
            WHEN number_of_rooms BETWEEN 101 AND 300 THEN '101-300'
            WHEN number_of_rooms BETWEEN 301 AND 500 THEN '301-500'
            ELSE '500+'
          END                                AS room_band,
          distributor_group,
          ROUND(SUM(spend_random_factor), 0) AS total_spend,
          COUNT(*)                           AS transaction_count
        FROM `{TABLE_REF}`
        WHERE distributor_group IS NOT NULL
        GROUP BY room_band, distributor_group
    """,
    "agg_category_drill": f"""
        SELECT
          category_level_1,
          category_level_2,
          COALESCE(category_level_3, '(none)') AS category_level_3,
          COALESCE(category_level_4, '(none)') AS category_level_4,
          ROUND(SUM(spend_random_factor), 0)   AS total_spend,
          COUNT(*)                             AS transaction_count
        FROM `{TABLE_REF}`
        GROUP BY category_level_1, category_level_2, category_level_3, category_level_4
    """,
}

for name, sql in views.items():
    full_name = f"{PROJECT_ID}.{DATASET_ID}.{name}"
    ddl = f"CREATE OR REPLACE VIEW `{full_name}` AS\n{sql.strip()}"
    client.query(ddl).result()
    print(f"  created  {full_name}")

print(f"\n{len(views)} views ready.")


  created  big-data-algorithms-493312.aramark_spend.agg_distributor_group
  created  big-data-algorithms-493312.aramark_spend.agg_category_level_1
  created  big-data-algorithms-493312.aramark_spend.agg_category_l1_l2
  created  big-data-algorithms-493312.aramark_spend.agg_state
  created  big-data-algorithms-493312.aramark_spend.agg_year_month
  created  big-data-algorithms-493312.aramark_spend.agg_business_entity_type
  created  big-data-algorithms-493312.aramark_spend.agg_market_segment
  created  big-data-algorithms-493312.aramark_spend.agg_room_band
  created  big-data-algorithms-493312.aramark_spend.agg_category_state
  created  big-data-algorithms-493312.aramark_spend.agg_category_month
  created  big-data-algorithms-493312.aramark_spend.agg_distributor_state
  created  big-data-algorithms-493312.aramark_spend.agg_category_entity
  created  big-data-algorithms-493312.aramark_spend.agg_distributor_room_band
  created  big-data-algorithms-493312.aramark_spend.agg_category_drill

1